In [ ]:
# Exploration ploidy <> redox
# PART 1 - Setup & settings

In [ ]:
import io, os, re, warnings
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
try:
  from IPython.display import display
except ImportError:
  display = print

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

os.environ["ROUTE_A_DATA"] = "/content/drive/MyDrive/Colab Notebooks/route_a_data"

Mounted at /content/drive


In [ ]:
LINEAGE = "All lineages" #@param ["All lineages", "CNS/Brain only"]
USE_PROTEINS = True #@param {type:"boolean"}
EXCLUDE_NRF2_MUTANTS = True #@param {type:"boolean"}
MY_GENES = "" #@param {type:"string"}
N_RANDOM_SETS = 500 #@param {type:"integer"}
N_PERMUTATIONS = 2000 #@param {type:"integer"}
MIN_LINES_PER_TISSUE = 10 #@param {type:"integer"}
DEPMAP_RELEASE = "DepMap Public 24Q4" #@param {type:"string"}
SEED = 1 #@param {type:"integer"}
GENES_OF_INTEREST = "GMNN, CDT1" #@param {type:"string"}
USE_CRISPR = True #@param {type:"boolean"}

# Reporting thresholds (conventions for labelling findings, not hypotheses)
FDR_MAX = 0.05 # false discovery rate
RANDOM_P_MAX = 0.05 # must beat expression-matched random gene sets
CONTROL_P_MAX = 0.05 # must beat every non-redox stress programme

DATA_DIR = os.environ.get("ROUTE_A_DATA", "/content/route_a_data")
os.makedirs(DATA_DIR, exist_ok=True)
rng = np.random.default_rng(SEED)
GOI = [g.strip().upper() for g in re.split(r"[,\s;]+", GENES_OF_INTEREST) if g.strip()]
print("Genes of interest:", GOI, "| data folder:", DATA_DIR)

Genes of interest: ['GMNN', 'CDT1'] | data folder: /content/drive/MyDrive/Colab Notebooks/route_a_data


In [ ]:
# Setup & check the data from my drive

# Refresh DATA_DIR to match the updated environment variable set in cell Bkkr_q7rPrqO
DATA_DIR = os.environ.get("ROUTE_A_DATA", "/content/route_a_data")
os.makedirs(DATA_DIR, exist_ok=True)
print("Searching in data folder:", DATA_DIR)

class StopHere(Exception):
  """Stops the notebook with a plain-language message."""

NEEDED = {
  "models": ["Model.csv"],
  "signatures": ["OmicsSignatures.csv"],
  "expression": ["OmicsExpressionProteinCodingGenesTPMLogp1.csv"],
  "damaging": ["OmicsSomaticMutationsMatrixDamaging.csv"],
  "hotspot": ["OmicsSomaticMutationsMatrixHotspot.csv"],
}
if USE_CRISPR:
  NEEDED["crispr"] = ["CRISPRGeneEffect.csv"]
if USE_PROTEINS:
  NEEDED["proteins"] = ["protein_quant_current_normalized.csv.gz", "protein_quant_current_normalized.csv"]

present = sorted(os.listdir(DATA_DIR))
paths, missing = {}, []
for key, names in NEEDED.items():
  found = next((n for n in names if os.path.exists(os.path.join(DATA_DIR, n))), None)
  if found:
    paths[key] = os.path.join(DATA_DIR, found)
    print(f" found {found} ({os.path.getsize(paths[key]) / 1e6:.0f} MB)")
  else:
    stem = names[0].split(".")[0]
    similar = [p for p in present if p.startswith(stem)]
    missing.append(names[0])
    print(f" MISSING {names[0]}" + (f" (similar name in folder: {similar[0]} - rename it)" if similar else ""))
if missing:
  raise StopHere(f"{len(missing)} file(s) missing from {DATA_DIR}. Add or rename them, then run this cell again. "
                "(Untick USE_CRISPR or USE_PROTEINS in Cell 1 to run without those files.)")
else:
  print("All files ready.")

Searching in data folder: /content/drive/MyDrive/Colab Notebooks/route_a_data
 found Model.csv (1 MB)
 found OmicsSignatures.csv (0 MB)
 found OmicsExpressionProteinCodingGenesTPMLogp1.csv (507 MB)
 found OmicsSomaticMutationsMatrixDamaging.csv (148 MB)
 found OmicsSomaticMutationsMatrixHotspot.csv (4 MB)
 found CRISPRGeneEffect.csv (429 MB)
 found protein_quant_current_normalized.csv.gz (31 MB)
All files ready.


In [ ]:
# Load and match the data

def read_model_matrix(path):
  """Read a DepMap cell-line x gene file (works with old and new layouts)."""
  table = pd.read_csv(path, low_memory=False)
  gene_columns = [c for c in table.columns if re.search(r"\(\d+\)\s*$", str(c))]
  if "ModelID" in table.columns:
    for flag in ("IsDefaultEntryForModel", "is_default_entry"):
      if flag in table.columns:
        table = table[table[flag].astype(str).str.lower().isin(["yes", "true", "1"])]
    table = table.drop_duplicates("ModelID").set_index("ModelID")
  else:
    table = table.set_index(table.columns[0])
  if not gene_columns:
    gene_columns = [c for c in table.columns if pd.api.types.is_numeric_dtype(table[c])]
  table = table[gene_columns].astype("float32")
  table.columns = [re.sub(r"\s*\(\d+\)\s*$", "", str(c)) for c in table.columns]
  table = table.loc[:, ~table.columns.duplicated()]
  table.index = table.index.astype(str)
  return table

def find_column(table, *names):
  lower = {str(c).lower(): c for c in table.columns}
  for name in names:
    if name.lower() in lower:
      return lower[name.lower()]
  raise StopHere(f"Expected a column like {names[0]}; found {list(table.columns)[:12]}")

models = pd.read_csv(paths["models"], low_memory=False)
models = models.set_index(find_column(models, "ModelID"))
models.index = models.index.astype(str)
lineage_col = find_column(models, "OncotreeLineage", "Lineage")
ccle_col = find_column(models, "CCLEName", "CCLE_Name")

signatures = pd.read_csv(paths["signatures"])
signatures = signatures.set_index("ModelID" if "ModelID" in signatures.columns else signatures.columns[0])
signatures.index = signatures.index.astype(str)

print("Reading expression (about 1 minute) ...")
expression = read_model_matrix(paths["expression"])
damaging = read_model_matrix(paths["damaging"])
hotspot = read_model_matrix(paths["hotspot"])

Reading expression (about 1 minute) ...


In [ ]:
def mutated(matrix, genes):
  present = [g for g in genes if g in matrix.columns]
  return (matrix[present].fillna(0) > 0).any(axis=1) if present else pd.Series(False, index=matrix.index)

nrf2_mutant = pd.concat([mutated(damaging, ["KEAP1", "CUL3"]), mutated(hotspot, ["NFE2L2", "KEAP1"])],
axis=1).any(axis=1)
lines = pd.DataFrame({
  "ploidy": pd.to_numeric(signatures[find_column(signatures, "Ploidy")], errors="coerce"),
  "wgd": pd.to_numeric(signatures[find_column(signatures, "WGD")], errors="coerce"),
}).dropna(subset=["ploidy"])
lines = lines[lines.index.isin(expression.index) & lines.index.isin(models.index)]
lines["tissue"] = models.loc[lines.index, lineage_col].astype(str)
lines["nrf2_mutant"] = nrf2_mutant.reindex(lines.index).fillna(False).astype(bool)
print(f"Cell lines with ploidy and expression: {len(lines)}")
if EXCLUDE_NRF2_MUTANTS:
  print(f"Leaving out {int(lines.nrf2_mutant.sum())} lines with KEAP1, CUL3 or NFE2L2 mutations")
  lines = lines[~lines.nrf2_mutant]
if LINEAGE == "CNS/Brain only":
  lines = lines[lines.tissue == "CNS/Brain"]
sizes = lines.tissue.value_counts()
lines = lines[lines.tissue.isin(sizes[sizes >= MIN_LINES_PER_TISSUE].index)]
print(f"Cell lines analysed: {len(lines)} from {lines.tissue.nunique()} tissues")
if len(lines) < 25:
  raise StopHere(f"Only {len(lines)} cell lines remain. Choose 'All lineages' in Cell 1 and rerun.")

proteins = None
if USE_PROTEINS:
  raw = pd.read_csv(paths["proteins"], low_memory=False)
  gene_col = find_column(raw, "Gene_Symbol", "GeneSymbol")
  sample_cols = [c for c in raw.columns if re.match(r"^.+_TenPx\d+$", str(c))]
  if not sample_cols:
    print("WARNING: protein file layout not recognised; proteins will be skipped.")
  by_gene = raw[[gene_col] + sample_cols].dropna(subset=[gene_col]).groupby(gene_col).mean()
  by_gene.columns = [re.sub(r"_TenPx\d+$", "", c) for c in by_gene.columns]
  by_gene = by_gene.T.groupby(level=0).mean()
  name_to_id = pd.Series(models.index, index=models[ccle_col].astype(str)).groupby(level=0).first()
  by_gene.index = by_gene.index.map(name_to_id)
  proteins = by_gene[by_gene.index.notna() & by_gene.index.isin(lines.index)]
  print(f"Cell lines that also have protein data: {len(proteins)}")

crispr = None
if USE_CRISPR:
  crispr = read_model_matrix(paths["crispr"])
  crispr = crispr[crispr.index.isin(lines.index)]
  print(f"Cell lines that also have CRISPR knockout data: {len(crispr)}")

Cell lines with ploidy and expression: 1303
Leaving out 47 lines with KEAP1, CUL3 or NFE2L2 mutations
Cell lines analysed: 1238 from 25 tissues
Cell lines that also have protein data: 266
Cell lines that also have CRISPR knockout data: 835


In [ ]:
# PART 2 - Scoring
# Each gene becomes a z-score (how far a line sits from the avergae line in standard deviations).
# This score is the average z-score of its genes. Genes measured in fewer than 80% of lines are left out.
# Before any correlation, tissue of origin and proliferation are removed from both ploidy and the score. What is left is how a line differs from other lines of the same tissue that divide equally fast.
# The correlation of those leftovers is the partial correlation (partial rho)

PROGRAMMES = {
    "REDOX: NRF2 targets": """NQO1 HMOX1 GCLC GCLM TXNRD1 SRXN1 G6PD PGD ME1 TALDO1 TKT FTH1 FTL ABCC1 SLC7A11 GSR PRDX1
TXN SQSTM1 AKR1C1 OSGIN1""",
    "REDOX: glutathione": """GSS GSR GPX1 GPX3 GPX4 GPX7 GPX8 GSTM1 GSTM2 GSTM3 GSTP1 GSTO1 GSTA4 GSTK1 MGST1 MGST3 GLRX
GLRX2 GLRX3 GCLC GCLM""",
    "REDOX: thioredoxin/peroxiredoxin": """TXN TXN2 TXNRD1 TXNRD2 TXNIP PRDX1 PRDX2 PRDX3 PRDX4 PRDX5 PRDX6 SRXN1 NXN
TXNDC5 TXNDC12""",
    "REDOX: antioxidant enzymes": """SOD1 SOD2 SOD3 CAT GPX1 GPX4 PRDX6 MSRA MSRB2 MSRB3 PARK7 NUDT1 OGG1 MUTYH APEX1""",
    "REDOX: ROS sources": """NOX4 NOX1 NOX5 CYBB CYBA NCF1 NCF2 NCF4 DUOX1 DUOX2 XDH AOX1 MAOA MAOB ALOX5 ALOX12
ALOX15""",
    "REDOX: ferroptosis": """ACSL4 LPCAT3 TFRC SLC7A11 GPX4 AIFM2 NCOA4 FTH1 FTL ALOX15 SLC3A2 VDAC2 VDAC3 PEBP1 DPP4""",
    "REDOX: mitochondrial respiration": """NDUFA1 NDUFA2 NDUFA4 NDUFB1 NDUFB5 NDUFS1 NDUFS2 NDUFV1 SDHA SDHB UQCRC1
UQCRC2 UQCRFS1 CYC1 COX4I1 COX5A COX5B COX6A1 COX7A2 ATP5F1A ATP5F1B ATP5MC1""",
    "CONTROL: unfolded protein response": """HSPA5 DDIT3 ATF4 XBP1 HERPUD1 DNAJB9 SEL1L EDEM1 CALR PDIA4 PDIA6 HSP90B1
ATF6 ERN1 EIF2AK3 SDF2L1 CRELD2 MANF""",
    "CONTROL: heat shock": """HSPA1A HSPA1B HSPB1 DNAJB1 HSPH1 HSPA8 HSP90AA1 HSP90AB1 HSPE1 HSPD1 BAG3 DNAJA1 HSF1
CRYAB""",
    "CONTROL: DNA damage": """CDKN1A GADD45A GADD45B MDM2 BBC3 BAX ATM ATR XPC DDB2 SESN1 RRM2B TP53I3 PPM1D""",
    "CONTROL: immediate-early genes": """FOS FOSB JUN JUNB EGR1 ARC NPAS4 NR4A1 DUSP1 IER2 ATF3 KLF4""",
}

PROLIFERATION = ("MKI67 TOP2A CCNB1 CDK1 BUB1 AURKA AURKB PLK1 CCNA2 MCM2 MCM3 MCM4 PCNA TYMS RRM2 "
                 "CDC20 KIF11 CENPF NUSAP1 UBE2C BIRC5 TPX2").split()

if MY_GENES.strip():
  PROGRAMMES["REDOX: my genes"] = " ".join(g.upper() for g in re.split(r"[,\s;]+", MY_GENES) if g)
PROGRAMMES = {n: [g for g in dict.fromkeys(v.split()) if g not in PROLIFERATION] for n, v in PROGRAMMES.items()}
print(sum(n.startswith("REDOX") for n in PROGRAMMES), "redox programmes,",
  sum(n.startswith("CONTROL") for n in PROGRAMMES), "stress controls")

7 redox programmes, 4 stress controls


In [ ]:
def zscore(frame):
  return (frame - frame.mean()) / frame.std(ddof=0).replace(0, np.nan)

def programme_scores(data, programmes, min_genes=5):
  """Each programme's score = average z-score of its genes measured in at least 80% of lines."""
  z = zscore(data[data.columns[data.notna().mean() >= 0.8]])
  scores, used = {}, {}
  for name, genes in programmes.items():
    found = [g for g in genes if g in z.columns]
    if len(found) >= min_genes:
      scores[name], used[name] = z[found].mean(axis=1, skipna=True), found
  return pd.DataFrame(scores), used

def adjustment_basis(table):
  """Tissue of origin + proliferation: the things every test removes first."""
  prolif = programme_scores(expression.loc[table.index], {"p": PROLIFERATION})[0]["p"].rank().to_numpy()
  dummies = pd.get_dummies(table["tissue"]).astype(float).to_numpy()
  q, _ = np.linalg.qr(np.column_stack([dummies, (prolif - prolif.mean()) / prolif.std()]))
  return q

def residual(values, basis):
  values = np.asarray(values, dtype=float)
  return values - basis @ (basis.T @ values)

def corr(a, b):
  a, b = a - a.mean(axis=0), b - b.mean(axis=0)
  return (a * b).sum(axis=0) / np.sqrt((a**2).sum(axis=0) * (b**2).sum(axis=0))

def shuffled_within_tissue(values, groups, n):
  out = np.tile(np.asarray(values, dtype=float)[:, None], (1, n))
  for g in np.unique(groups):
    idx = np.flatnonzero(groups == g)
    for j in range(n):
      out[idx, j] = out[rng.permutation(idx), j]
  return out

def perm_p(observed, null):
  return (1 + np.sum(np.abs(null) >= abs(observed))) / (len(null) + 1)

def fdr(p):
  p = np.asarray(p, dtype=float)
  order = np.argsort(p)
  adjusted = p[order] * len(p) / np.arange(1, len(p) + 1)
  out = np.empty_like(p)
  out[order] = np.minimum(np.minimum.accumulate(adjusted[::-1])[::-1], 1)
  return out
print("Tools ready.")

Tools ready.


In [ ]:
# Which programmes go with ploidy?

def test_link(data, label, exposure="ploidy"):
  """Partial rank correlation of programme score with ploidy, after removing tissue and proliferation.
  The p-value comes from shuffling ploidy within each tissue N_PERMUTATIONS times."""
  table = lines[lines.index.isin(data.index)].dropna(subset=[exposure])
  data = data.loc[table.index]
  basis = adjustment_basis(table)
  x = residual(table[exposure].rank().to_numpy(), basis)
  xs = residual(shuffled_within_tissue(table[exposure].rank().to_numpy(), table["tissue"].to_numpy(),
                                      N_PERMUTATIONS), basis)
  scores, used = programme_scores(data, PROGRAMMES)
  rows = []
  for name in scores.columns:
    s = scores[name].to_numpy()
    ok = ~np.isnan(s)
    r = residual(np.where(ok, s, np.nanmean(s)), basis)
    rho = float(corr(x[ok, None], r[ok, None])[0])
    null = corr(xs[ok], np.repeat(r[ok, None], xs.shape[1], axis=1))
    rows.append({"data": label, "programme": name, "lines": int(ok.sum()), "genes_used": len(used[name]),
                "partial_rho": rho, "p_perm": perm_p(rho, null)})
  result = pd.DataFrame(rows)
  redox = result.programme.str.startswith("REDOX")
  result.loc[redox, "q_fdr"] = fdr(result.loc[redox, "p_perm"])
  return {"result": result, "scores": scores, "used": used, "x": x, "xs": xs, "basis": basis,
          "table": table, "data": data}

rna = test_link(expression, "RNA")
display(rna["result"].round(3))

,data,programme,lines,genes_used,partial_rho,p_perm,q_fdr
0,RNA,REDOX: NRF2 targets,1238,21,-0.010,0.714,0.981
1,RNA,REDOX: glutathione,1238,21,-0.005,0.874,0.981
2,RNA,REDOX: thioredoxin/peroxiredoxin,1238,15,-0.012,0.675,0.981
3,RNA,REDOX: antioxidant enzymes,1238,15,-0.071,0.015,0.108
4,RNA,REDOX: ROS sources,1238,17,-0.032,0.267,0.936
5,RNA,REDOX: ferroptosis,1238,15,-0.019,0.517,0.981
6,RNA,REDOX: mitochondrial respiration,1238,22,-0.001,0.981,0.981
7,RNA,CONTROL: unfolded protein response,1238,18,0.032,0.270,NaN
8,RNA,CONTROL: heat shock,1238,14,-0.002,0.935,NaN
9,RNA,CONTROL: DNA damage,1238,14,-0.112,0.000,NaN


In [ ]:
# Is it stronger than random genes?

def test_random(ctx):
  """For each redox programme, draw N_RANDOM_SETS random gene sets of the same size, each gene matched
  on average expression, and see how strongly they correlate with ploidy."""
  data, used, x, basis, result = ctx["data"], ctx["used"], ctx["x"], ctx["basis"], ctx["result"]
  usable = data.columns[data.notna().mean() >= 0.8]
  level = data[usable].mean()
  bins = pd.qcut(level.rank(method="first"), 20, labels=False)
  avoid = set().union(*used.values()) | set(PROLIFERATION)
  pool = [g for g in usable if g not in avoid]
  by_bin = {b: [g for g in level.index[bins == b] if g not in avoid] or pool for b in range(20)}
  z = zscore(data[usable])
  for name, genes in used.items():
    if not name.startswith("REDOX"):
      continue
    null = []
    for _ in range(N_RANDOM_SETS):
      draw = [rng.choice(by_bin[int(bins[g])]) for g in genes]
      null.append(corr(x[:, None], residual(z[draw].mean(axis=1).fillna(0).to_numpy(), basis)[:, None])[0])
    i = result.programme == name
    rho = float(result.loc[i, "partial_rho"].iloc[0])
    result.loc[i, "random_low"], result.loc[i, "random_high"] = np.quantile(null, [0.025, 0.975])
    result.loc[i, "p_vs_random"] = perm_p(rho, np.array(null))

test_random(rna)
display(rna["result"].loc[rna["result"].programme.str.startswith("REDOX"),
        ["programme", "partial_rho", "random_low", "random_high", "p_vs_random"]].round(3))

,programme,partial_rho,random_low,random_high,p_vs_random
0,REDOX: NRF2 targets,-0.010,-0.069,0.042,0.754
1,REDOX: glutathione,-0.005,-0.063,0.049,0.878
2,REDOX: thioredoxin/peroxiredoxin,-0.012,-0.071,0.053,0.719
3,REDOX: antioxidant enzymes,-0.071,-0.076,0.056,0.044
4,REDOX: ROS sources,-0.032,-0.066,0.067,0.341
5,REDOX: ferroptosis,-0.019,-0.075,0.049,0.621
6,REDOX: mitochondrial respiration,-0.001,-0.069,0.047,0.986


In [ ]:
# Is it specific to redox, or general stress?

def test_controls(ctx):
  """For each redox programme and each stress control, test whether (redox minus control) still goes with
  ploidy. The worst (largest) p-value across the four controls is kept."""
  scores, x, xs, basis, result = ctx["scores"], ctx["x"], ctx["xs"], ctx["basis"], ctx["result"]
  controls = [c for c in scores.columns if c.startswith("CONTROL")]
  for name in [n for n in scores.columns if n.startswith("REDOX")]:
    i = result.programme == name
    rho = float(result.loc[i, "partial_rho"].iloc[0])
    worst_p, worst = 0.0, ""
    for control in controls:
      diff = zscore(scores[[name]])[name] - zscore(scores[[control]])[control]
      r = residual(diff.fillna(0).to_numpy(), basis)
      observed = float(corr(x[:, None], r[:, None])[0])
      p = perm_p(observed, corr(xs, np.repeat(r[:, None], xs.shape[1], axis=1)))
      if np.sign(observed) != np.sign(rho):
        p = 1.0
      if p >= worst_p:
        worst_p, worst = p, control.replace("CONTROL: ", "")
    result.loc[i, "p_vs_controls"], result.loc[i, "closest_control"] = worst_p, worst

test_controls(rna)
display(rna["result"][["programme", "partial_rho"]].round(3).join(
  rna["result"][["p_vs_controls", "closest_control"]]))

,programme,partial_rho,p_vs_controls,closest_control
0,REDOX: NRF2 targets,-0.010,1.0,DNA damage
1,REDOX: glutathione,-0.005,1.0,DNA damage
2,REDOX: thioredoxin/peroxiredoxin,-0.012,1.0,DNA damage
3,REDOX: antioxidant enzymes,-0.071,1.0,DNA damage
4,REDOX: ROS sources,-0.032,1.0,DNA damage
5,REDOX: ferroptosis,-0.019,1.0,DNA damage
6,REDOX: mitochondrial respiration,-0.001,1.0,DNA damage
7,CONTROL: unfolded protein response,0.032,NaN,NaN
8,CONTROL: heat shock,-0.002,NaN,NaN
9,CONTROL: DNA damage,-0.112,NaN,NaN


In [ ]:
wgd = test_link(expression, "RNA, whole-genome doubling", exposure="wgd")
test_random(wgd)
test_controls(wgd)
print(f"{int(wgd['table'].wgd.sum())} of {len(wgd['table'])} lines have a doubled genome")
display(wgd["result"].loc[wgd["result"].programme.str.startswith("REDOX"),
                          ["programme", "partial_rho", "q_fdr", "p_vs_random", "p_vs_controls"]].round(3))

print(wgd["table"].wgd.value_counts().sort_index())

834 of 1238 lines have a doubled genome


,programme,partial_rho,q_fdr,p_vs_random,p_vs_controls
0,REDOX: NRF2 targets,0.004,0.898,0.920,1.0
1,REDOX: glutathione,-0.013,0.898,0.703,1.0
2,REDOX: thioredoxin/peroxiredoxin,0.004,0.898,0.930,1.0
3,REDOX: antioxidant enzymes,-0.072,0.105,0.106,1.0
4,REDOX: ROS sources,-0.024,0.898,0.613,1.0
5,REDOX: ferroptosis,0.010,0.898,0.846,1.0
6,REDOX: mitochondrial respiration,-0.026,0.898,0.539,1.0


wgd
0.0    404
1.0    834
Name: count, dtype: int64


In [ ]:
# Proteins

prot = None
if proteins is not None and len(proteins) >= 25:
  prot = test_link(proteins, "Protein")
  test_random(prot)
  test_controls(prot)
  display(prot["result"].loc[prot["result"].programme.str.startswith("REDOX"),
                              ["programme", "lines", "genes_used", "partial_rho", "q_fdr", "p_vs_random",
                              "p_vs_controls"]].round(3))
else:
  print("Proteins skipped: switched off in Cell 1 or fewer than 25 matching cell lines.")

,programme,lines,genes_used,partial_rho,q_fdr,p_vs_random,p_vs_controls
0,REDOX: NRF2 targets,266,20,-0.036,0.696,0.549,1.000
1,REDOX: glutathione,266,19,-0.042,0.696,0.475,1.000
2,REDOX: thioredoxin/peroxiredoxin,266,15,-0.055,0.696,0.349,1.000
3,REDOX: antioxidant enzymes,266,11,0.057,0.696,0.297,1.000
4,REDOX: ferroptosis,266,14,-0.092,0.696,0.100,0.781
5,REDOX: mitochondrial respiration,266,18,-0.004,0.959,0.944,1.000


In [ ]:
# Gene by gene

def gene_screen(ctx, label):
  """Every redox and control gene, plus GENES_OF_INTEREST, tested on its own against ploidy (same adjustment
  and shuffles as Screen 1). False discovery rate is computed across all genes tested."""
  data, x, xs, basis = ctx["data"], ctx["x"], ctx["xs"], ctx["basis"]
  genes = list(dict.fromkeys([g for v in PROGRAMMES.values() for g in v] + GOI))
  genes = [g for g in genes if g in data.columns and data[g].notna().mean() >= 0.8]
  z = zscore(data[genes])
  R = residual(z.fillna(z.mean()).fillna(0).to_numpy(), basis)
  R = (R - R.mean(0)) / np.where(R.std(0) > 0, R.std(0), 1)
  xz = (x - x.mean()) / x.std()
  xsz = (xs - xs.mean(0)) / xs.std(0)
  rho = xz @ R / len(xz)
  null = xsz.T @ R / len(xz)
  p = (1 + (np.abs(null) >= np.abs(rho)).sum(0)) / (null.shape[0] + 1)
  member = {g: ", ".join(n.split(": ")[1] for n, v in PROGRAMMES.items() if g in v) for g in genes}
  out = pd.DataFrame({"data": label, "gene": genes, "partial_rho": rho, "p_perm": p,
                      "programmes": [member[g] or "gene of interest" for g in genes]})
  out["q_fdr"] = fdr(out["p_perm"])
  return out.assign(_abs=out.partial_rho.abs()).sort_values(["p_perm", "_abs"], ascending=[True,
False]).drop(columns="_abs")
gene_results = [gene_screen(rna, "RNA")]
if prot is not None:
  gene_results.append(gene_screen(prot, "Protein"))
gene_results = pd.concat(gene_results, ignore_index=True)
both = gene_results.pivot_table(index="gene", columns="data", values=["partial_rho", "q_fdr"])
print("Top 15 genes in RNA:")
display(gene_results[gene_results.data == "RNA"].head(15).round(3))
if prot is not None:
  agree = both[(both[("q_fdr", "RNA")] < FDR_MAX) & (both[("q_fdr", "Protein")] < FDR_MAX) &
              (np.sign(both[("partial_rho", "RNA")]) == np.sign(both[("partial_rho", "Protein")]))]
  print(f"\nGenes significant in both RNA and protein, same direction: {len(agree)}")
  display(agree.round(3))

Top 15 genes in RNA:


,data,gene,partial_rho,p_perm,programmes,q_fdr
0,RNA,PRDX3,-0.120,0.000,thioredoxin/peroxiredoxin,0.017
1,RNA,ABCC1,-0.104,0.000,NRF2 targets,0.017
2,RNA,CDKN1A,-0.102,0.000,DNA damage,0.017
3,RNA,PRDX1,-0.101,0.000,"NRF2 targets, thioredoxin/peroxiredoxin",0.017
4,RNA,ME1,0.099,0.000,NRF2 targets,0.017
5,RNA,BAX,-0.104,0.001,DNA damage,0.021
6,RNA,VDAC2,-0.104,0.001,ferroptosis,0.021
7,RNA,RRM2B,-0.102,0.001,DNA damage,0.021
8,RNA,SESN1,-0.100,0.001,DNA damage,0.023
9,RNA,MSRA,-0.096,0.001,antioxidant enzymes,0.023



Genes significant in both RNA and protein, same direction: 0


Empty DataFrame
Columns: [(partial_rho, Protein), (partial_rho, RNA), (q_fdr, Protein), (q_fdr, RNA)]
Index: []

In [ ]:
# Geminin

def partial_assoc(table, a, scores, extra=None, n_perm=N_PERMUTATIONS):
  """Partial rank correlation of vector a with each column of scores, within tissue, adjusted for
  proliferation (and optionally ploidy); p-value by shuffling a within tissue."""
  cols = [adjustment_basis(table)]
  if extra is not None:
    cols.append(np.asarray(extra, dtype=float)[:, None])
  basis, _ = np.linalg.qr(np.column_stack(cols))
  a_rank = pd.Series(a).rank().to_numpy()
  xa = residual(a_rank, basis)
  xsa = residual(shuffled_within_tissue(a_rank, table["tissue"].to_numpy(), n_perm), basis)
  rows = []
  for name in scores.columns:
    s = scores[name].to_numpy()
    r = residual(np.where(np.isnan(s), np.nanmean(s), s), basis)
    rho = float(corr(xa[:, None], r[:, None])[0])
    rows.append({"against": name, "partial_rho": rho,
                "p_perm": perm_p(rho, corr(xsa, np.repeat(r[:, None], xsa.shape[1], axis=1)))})
  out = pd.DataFrame(rows)
  out["q_fdr"] = fdr(out["p_perm"])
  return out

def rho_only(table, A, scores, extra=None):
  """Partial correlations of many vectors (columns of A) with each score, no shuffling (fast)."""
  cols = [adjustment_basis(table)]
  if extra is not None:
    cols.append(np.asarray(extra, dtype=float)[:, None])
  basis, _ = np.linalg.qr(np.column_stack(cols))
  RA = residual(pd.DataFrame(A).rank().to_numpy(), basis)
  S = scores.to_numpy(float)
  RS = residual(np.where(np.isnan(S), np.nanmean(S, axis=0), S), basis)
  RA = (RA - RA.mean(0)) / RA.std(0)
  RS = (RS - RS.mean(0)) / RS.std(0)
  return RA.T @ RS / len(RA)

def matched_genes(gene, table, n=200):
  """Genes like `gene` in average expression and in how closely they follow proliferation."""
  data = expression.loc[table.index]
  avoid = set(g for v in PROGRAMMES.values() for g in v) | set(PROLIFERATION) | set(GOI)
  pool = [g for g in data.columns if g not in avoid and data[g].std() > 0]
  prolif = programme_scores(data, {"p": PROLIFERATION})[0]["p"].rank()
  level = data[pool + [gene]].mean()
  follow = data[pool + [gene]].rank().corrwith(prolif)
  lb = pd.qcut(level.rank(method="first"), 5, labels=False)
  fb = pd.qcut(follow.rank(method="first"), 5, labels=False)
  same = [g for g in pool if lb[g] == lb[gene] and fb[g] == fb[gene]]
  return list(rng.choice(same, min(n, len(same)), replace=False))

table = rna["table"]
measures = {}
for g in GOI:
  if g in expression.columns:
    measures[f"{g} RNA"] = expression.loc[table.index, g].to_numpy(float)
  if crispr is not None and g in crispr.columns:
    measures[f"{g} knockout effect"] = crispr[g].reindex(table.index).to_numpy(float)
if {"GMNN", "CDT1"} <= set(expression.columns):
  measures["GMNN minus CDT1 (RNA)"] = (expression.loc[table.index, "GMNN"] - expression.loc[table.index,
"CDT1"]).to_numpy(float)

redox_scores = rna["scores"][[c for c in rna["scores"].columns if c.startswith("REDOX")]]

geminin = []
for label, values in measures.items():
  ok = ~np.isnan(values)
  if ok.sum() < 25:
    continue
  sub = table[ok]
  ploidy_scores = pd.DataFrame({"ploidy": sub["ploidy"].to_numpy(), "whole-genome doubling": sub["wgd"].fillna(0).to_numpy()})
  a = partial_assoc(sub, values[ok], ploidy_scores).assign(measure=label, adjusted_for_ploidy="-")
  b = partial_assoc(sub, values[ok], redox_scores.loc[sub.index].reset_index(drop=True)).assign(measure=label, adjusted_for_ploidy="no")
  c = partial_assoc(sub, values[ok], redox_scores.loc[sub.index].reset_index(drop=True),
           extra=sub["ploidy"].rank().to_numpy()).assign(measure=label, adjusted_for_ploidy="yes")
  part = pd.concat([a, b, c], ignore_index=True).assign(lines=int(ok.sum()))
  gene = label.split()[0]
  if label.endswith(" RNA") and gene in expression.columns:
    others = matched_genes(gene, sub)
    A = expression.loc[sub.index, others].to_numpy(float)
    targets = pd.concat([ploidy_scores, redox_scores.loc[sub.index].reset_index(drop=True)], axis=1)
    null_no = rho_only(sub, A, targets)
    null_yes = rho_only(sub, A, redox_scores.loc[sub.index].reset_index(drop=True), sub["ploidy"].rank().to_numpy())
    for i, row in part.reset_index(drop=True).iterrows():
      null = (null_yes[:, list(redox_scores.columns).index(row["against"])] if row["adjusted_for_ploidy"] == "yes"
          else null_no[:, list(targets.columns).index(row["against"])])
      part.loc[part.index[i], "matched_low"] = np.quantile(null, 0.025)
      part.loc[part.index[i], "matched_high"] = np.quantile(null, 0.975)
      part.loc[part.index[i], "p_vs_matched_genes"] = perm_p(row["partial_rho"], null)
  geminin.append(part)

geminin = pd.concat(geminin, ignore_index=True) if geminin else pd.DataFrame()
for c in ("matched_low", "matched_high", "p_vs_matched_genes"):
  if not geminin.empty and c not in geminin.columns:
    geminin[c] = np.nan
if geminin.empty:
  print("None of the genes of interest were found.")
else:
  for label in measures:
    part = geminin[geminin.measure == label]
    if part.empty:
      continue
    print(f"\n{label} ({part.lines.iloc[0]} lines)")
    cols = ["against", "adjusted_for_ploidy", "partial_rho", "p_perm", "q_fdr"]
    cols += [c for c in ("matched_low", "matched_high", "p_vs_matched_genes") if part[c].notna().any()]
    display(part[cols].round(3).reset_index(drop=True))


GMNN RNA (1238 lines)


,against,adjusted_for_ploidy,partial_rho,p_perm,q_fdr,matched_low,matched_high,p_vs_matched_genes
0,ploidy,-,0.051,0.073,0.073,-0.104,0.088,0.328
1,whole-genome doubling,-,0.054,0.056,0.073,-0.134,0.102,0.423
2,REDOX: NRF2 targets,no,-0.069,0.019,0.059,-0.103,0.279,0.642
3,REDOX: glutathione,no,0.026,0.368,0.430,-0.062,0.225,0.846
4,REDOX: thioredoxin/peroxiredoxin,no,0.019,0.510,0.510,-0.081,0.319,0.915
5,REDOX: antioxidant enzymes,no,-0.035,0.217,0.304,-0.024,0.310,0.940
6,REDOX: ROS sources,no,-0.063,0.025,0.059,-0.119,0.187,0.527
7,REDOX: ferroptosis,no,0.038,0.201,0.304,-0.085,0.316,0.866
8,REDOX: mitochondrial respiration,no,0.202,0.000,0.003,-0.082,0.332,0.194
9,REDOX: NRF2 targets,yes,-0.069,0.018,0.063,-0.103,0.279,0.642



GMNN knockout effect (835 lines)


,against,adjusted_for_ploidy,partial_rho,p_perm,q_fdr
0,ploidy,-,0.011,0.748,0.748
1,whole-genome doubling,-,-0.012,0.727,0.748
2,REDOX: NRF2 targets,no,-0.107,0.002,0.017
3,REDOX: glutathione,no,-0.026,0.467,0.818
4,REDOX: thioredoxin/peroxiredoxin,no,-0.072,0.044,0.156
5,REDOX: antioxidant enzymes,no,0.001,0.973,0.973
6,REDOX: ROS sources,no,-0.008,0.807,0.941
7,REDOX: ferroptosis,no,-0.041,0.244,0.570
8,REDOX: mitochondrial respiration,no,0.014,0.708,0.941
9,REDOX: NRF2 targets,yes,-0.107,0.002,0.017



CDT1 RNA (1238 lines)


,against,adjusted_for_ploidy,partial_rho,p_perm,q_fdr,matched_low,matched_high,p_vs_matched_genes
0,ploidy,-,-0.038,0.189,0.378,-0.096,0.072,0.358
1,whole-genome doubling,-,-0.016,0.579,0.579,-0.126,0.077,0.781
2,REDOX: NRF2 targets,no,0.005,0.866,0.866,-0.093,0.215,0.945
3,REDOX: glutathione,no,-0.035,0.228,0.319,-0.077,0.224,0.721
4,REDOX: thioredoxin/peroxiredoxin,no,-0.018,0.534,0.623,-0.071,0.244,0.896
5,REDOX: antioxidant enzymes,no,0.066,0.021,0.115,-0.031,0.286,0.786
6,REDOX: ROS sources,no,0.063,0.033,0.115,-0.124,0.208,0.582
7,REDOX: ferroptosis,no,-0.042,0.145,0.254,-0.098,0.293,0.806
8,REDOX: mitochondrial respiration,no,0.050,0.080,0.187,-0.071,0.241,0.697
9,REDOX: NRF2 targets,yes,0.005,0.865,0.865,-0.093,0.215,0.955



CDT1 knockout effect (835 lines)


,against,adjusted_for_ploidy,partial_rho,p_perm,q_fdr
0,ploidy,-,0.144,0.000,0.001
1,whole-genome doubling,-,0.120,0.001,0.001
2,REDOX: NRF2 targets,no,0.004,0.915,0.954
3,REDOX: glutathione,no,-0.020,0.558,0.781
4,REDOX: thioredoxin/peroxiredoxin,no,-0.048,0.159,0.556
5,REDOX: antioxidant enzymes,no,-0.054,0.114,0.556
6,REDOX: ROS sources,no,-0.032,0.343,0.601
7,REDOX: ferroptosis,no,-0.037,0.277,0.601
8,REDOX: mitochondrial respiration,no,-0.002,0.954,0.954
9,REDOX: NRF2 targets,yes,0.000,0.993,0.993



GMNN minus CDT1 (RNA) (1238 lines)


,against,adjusted_for_ploidy,partial_rho,p_perm,q_fdr
0,ploidy,-,0.051,0.089,0.158
1,whole-genome doubling,-,0.041,0.158,0.158
2,REDOX: NRF2 targets,no,-0.024,0.423,0.423
3,REDOX: glutathione,no,0.066,0.020,0.036
4,REDOX: thioredoxin/peroxiredoxin,no,0.054,0.067,0.094
5,REDOX: antioxidant enzymes,no,-0.045,0.121,0.141
6,REDOX: ROS sources,no,-0.101,0.001,0.003
7,REDOX: ferroptosis,no,0.096,0.000,0.003
8,REDOX: mitochondrial respiration,no,0.095,0.001,0.003
9,REDOX: NRF2 targets,yes,-0.023,0.441,0.441


In [ ]:
# How unusual is GMNN? Correlation of every gene's knockout effect with the NRF2 targets score

sub = table[table.index.isin(crispr.index)]
ko = crispr.loc[sub.index]
ko = ko.loc[:, ko.notna().mean() >= 0.9]
A = ko.fillna(ko.median()).to_numpy(float)
nrf2 = pd.DataFrame({"nrf2": rna["scores"].loc[sub.index, "REDOX: NRF2 targets"].to_numpy()})
r = pd.Series(rho_only(sub, A, nrf2)[:, 0], index=ko.columns)
print(f"{len(r)} genes. Knockout effect vs NRF2 targets score:")
print(f"  median {r.median():+.3f}, middle 95% from {r.quantile(0.025):+.3f} to {r.quantile(0.975):+.3f}")
for g in ["GMNN", "CDT1"]:
    if g in r.index:
        print(f"  {g}: {r[g]:+.3f}, more negative than {100 * (r > r[g]).mean():.1f}% of genes")
print("\nThe 15 most negative genes:")
print(r.sort_values().head(15).round(3))

17787 genes. Knockout effect vs NRF2 targets score:
  median -0.001, middle 95% from -0.086 to +0.084
  GMNN: -0.107, more negative than 99.2% of genes
  CDT1: +0.004, more negative than 44.9% of genes

The 15 most negative genes:
NFE2L2     -0.337
TAPT1      -0.301
SUCO       -0.291
TGIF1      -0.213
SLC33A1    -0.199
ITGAV      -0.187
UXS1       -0.184
ARF4       -0.173
COPG1      -0.172
DCAF11     -0.168
TLN1       -0.168
CCND3      -0.165
C19orf53   -0.160
MMACHC     -0.159
CHMP5      -0.157
dtype: float64


In [ ]:
# Do NRF2-pathway-mutant lines (left out so far) depend more on Geminin?
ko_all = read_model_matrix(paths["crispr"])["GMNN"].dropna()
df = pd.DataFrame({"gmnn_ko": ko_all})
df["tissue"] = models[lineage_col].reindex(df.index).astype(str)
df["nrf2_mutant"] = nrf2_mutant.reindex(df.index).fillna(False).astype(bool)
ok = df.groupby("tissue")["nrf2_mutant"].agg(lambda s: s.sum() >= 3 and (~s).sum() >= 3)
df = df[df.tissue.isin(ok[ok].index)].copy()
df["centred"] = df.gmnn_ko - df.groupby("tissue")["gmnn_ko"].transform("median")
diff = lambda m: df.loc[m, "centred"].mean() - df.loc[~m, "centred"].mean()
obs = diff(df.nrf2_mutant.to_numpy())
null = []
for _ in range(2000):
    shuffled = df.groupby("tissue")["nrf2_mutant"].transform(lambda s: rng.permutation(s.to_numpy())).to_numpy(bool)
    null.append(diff(shuffled))
p = (1 + np.sum(np.abs(null) >= abs(obs))) / (len(null) + 1)
print(df.groupby("tissue")["nrf2_mutant"].agg(mutant="sum", total="size"))
print(f"\nGMNN knockout effect, mutant minus non-mutant (within tissue): {obs:+.3f}, p = {p:.3f}")
print("Negative = NRF2-mutant lines depend MORE on Geminin")

                       mutant  total
tissue                              
Bladder/Urinary Tract       3     34
Esophagus/Stomach           7     69
Head and Neck               4     77
Lung                       17    126
Uterus                      5     34

GMNN knockout effect, mutant minus non-mutant (within tissue): -0.006, p = 0.907
Negative = NRF2-mutant lines depend MORE on Geminin
